# Experiment 4 — Symmetry & Transitivity Check

**PDF reference: Phase 4, Experiment 4**

A robust analysis should satisfy two internal consistency checks:

1. **Symmetry**: `delta(A→B) + delta(B→A) ≈ 0`  
   If you measure B as 1% faster than A, then measuring A vs B should give A as ~1% slower.

2. **Transitivity** (3+ bikes): `delta(A→C) ≈ delta(A→B) + delta(B→C)`  
   The indirect route through B should agree with the direct comparison.

Both checks are done by re-running Phases 2–4 with each bike as the reference.

**Also included:** Grade decomposition — delta vs segment grade scatterplot to see
whether the effect is concentrated on flat (aero) or climbing (weight) segments.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats

from src.database import init_db, load_efforts, load_segments, load_bikes
from src.bike_delta import (
    prepare_delta_dataset, get_paired_segments,
    fit_baseline_model, compute_residuals,
    per_segment_delta, weighted_delta_summary,
)
from src.analytics import filter_outliers_by_power_speed

sns.set_theme(style='whitegrid', palette='Set2')
print('Imports OK')

In [ ]:
# ── Load and prepare ──────────────────────────────────────────────────────────
init_db()
raw_efforts = load_efforts()
segments = load_segments()
bikes_df = load_bikes()
bikes_dict = dict(zip(bikes_df['gear_id'], bikes_df['name'])) if not bikes_df.empty else {}

power_efforts = raw_efforts[raw_efforts['average_watts'].notna()].copy()
df = prepare_delta_dataset(power_efforts, segments, bikes_dict)
df, _ = filter_outliers_by_power_speed(df, z_threshold=2.0)

all_bikes = df['bike_name'].dropna().unique().tolist()

# Configure
SPLINE_DF = 5
MIN_EFFORTS = 3

paired = get_paired_segments(df, all_bikes, min_efforts=MIN_EFFORTS)
print(f'All bikes: {all_bikes}')
print(f'Paired segments: {len(paired)}')

## 4.1 — Full symmetry check
Run the pipeline with each bike as reference, collect all pairwise deltas, then verify `delta(A→B) + delta(B→A) ≈ 0`.

In [ ]:
# Store (ref_bike, other_bike) → delta
delta_matrix: dict[tuple[str, str], float] = {}
se_matrix: dict[tuple[str, str], float] = {}

for ref in all_bikes:
    print(f'Fitting baseline on {ref}…')
    try:
        m, di = fit_baseline_model(df, ref, spline_df=SPLINE_DF)
        dr = compute_residuals(df, m, di)
        deltas = per_segment_delta(dr, paired, ref, all_bikes)
        summary = weighted_delta_summary(deltas)
        for _, row in summary.iterrows():
            # pair format: "A → B"
            parts = str(row['bike_pair']).split(' → ')
            if len(parts) == 2:
                ref_name, other_name = parts[0].strip(), parts[1].strip()
                delta_matrix[(ref_name, other_name)] = float(row['delta'])
                se_matrix[(ref_name, other_name)] = float(row['se'])
    except Exception as e:
        print(f'  ERROR for ref={ref}: {e}')

print('\nDone.')
print(f'Delta estimates collected: {len(delta_matrix)}')

In [ ]:
# ── Symmetry table: delta(A→B) + delta(B→A) ≈ 0 ──────────────────────────────
sym_rows = []
for a, b in itertools.combinations(all_bikes, 2):
    d_ab = delta_matrix.get((a, b), None)
    d_ba = delta_matrix.get((b, a), None)
    if d_ab is not None and d_ba is not None:
        total = d_ab + d_ba
        sym_rows.append({
            'Pair': f'{a} ↔ {b}',
            f'δ({a}→{b})': f'{d_ab:+.5f}',
            f'δ({b}→{a})': f'{d_ba:+.5f}',
            'Sum': f'{total:+.5f}',
            'Symmetric?': '✅' if abs(total) < 0.01 else '⚠️'
        })
    else:
        sym_rows.append({
            'Pair': f'{a} ↔ {b}',
            f'δ({a}→{b})': str(d_ab),
            f'δ({b}→{a})': str(d_ba),
            'Sum': 'N/A',
            'Symmetric?': '❓'
        })

sym_df = pd.DataFrame(sym_rows)
print('=== Symmetry check (|sum| < 0.01 is ideal) ===')
print(sym_df.to_string(index=False))

## 4.2 — Transitivity check (3+ bikes)
For every triplet (A, B, C): `delta(A→C) ≈ delta(A→B) + delta(B→C)`

In [ ]:
if len(all_bikes) < 3:
    print('Need ≥ 3 bikes for transitivity check. Only', len(all_bikes), 'found.')
else:
    trans_rows = []
    for a, b, c in itertools.permutations(all_bikes, 3):
        d_ac = delta_matrix.get((a, c), None)
        d_ab = delta_matrix.get((a, b), None)
        d_bc = delta_matrix.get((b, c), None)
        if all(v is not None for v in [d_ac, d_ab, d_bc]):
            indirect = d_ab + d_bc
            err = d_ac - indirect
            trans_rows.append({
                'Direct path': f'{a}→{c}',
                'Indirect path': f'{a}→{b}→{c}',
                'δ(direct)': f'{d_ac:+.5f}',
                'δ(A→B) + δ(B→C)': f'{indirect:+.5f}',
                'Error': f'{err:+.5f}',
                'Transitive?': '✅' if abs(err) < 0.01 else '⚠️'
            })

    if trans_rows:
        trans_df = pd.DataFrame(trans_rows)
        print('=== Transitivity check ===')
        print(trans_df.to_string(index=False))
    else:
        print('Could not compute any transitivity triples — check delta_matrix.')

## 4.3 — Grade decomposition
Is the speed delta concentrated on flat segments (aero effect) or steep ones (weight effect)?  
Scatter: per-segment delta vs segment grade, with a simple regression trendline.

In [ ]:
# Use first reference bike to get a per-segment delta table with grade info
REF_BIKE = all_bikes[0]
m0, di0 = fit_baseline_model(df, REF_BIKE, spline_df=SPLINE_DF)
dr0 = compute_residuals(df, m0, di0)
deltas0 = per_segment_delta(dr0, paired, REF_BIKE, all_bikes)

other_bikes = [b for b in all_bikes if b != REF_BIKE]
palette = sns.color_palette('Set2', n_colors=len(other_bikes))

if 'grade' in deltas0.columns and not deltas0.empty:
    fig, axes = plt.subplots(1, len(other_bikes), figsize=(6 * len(other_bikes), 4), squeeze=False)
    for idx, other in enumerate(other_bikes):
        ax = axes[0][idx]
        pair_data = deltas0[deltas0['other_bike'] == other].dropna(subset=['grade', 'delta'])
        if pair_data.empty:
            ax.set_title(f'{REF_BIKE} → {other}\n(no data)')
            continue

        ax.scatter(pair_data['grade'], pair_data['delta'],
                   color=palette[idx], alpha=0.7, s=60, label='Segment delta')

        # OLS trendline
        if len(pair_data) >= 3:
            slope, intercept, r, p, se = scipy_stats.linregress(
                pair_data['grade'], pair_data['delta']
            )
            x_fit = np.linspace(pair_data['grade'].min(), pair_data['grade'].max(), 100)
            ax.plot(x_fit, slope * x_fit + intercept, 'k-', lw=2,
                    label=f'OLS  r={r:.2f}  p={p:.3f}')

        ax.axhline(0, color='grey', linestyle='--', lw=1)
        ax.set_xlabel('Segment average grade (%)')
        ax.set_ylabel('Delta (speed/W¹⁄³)')
        ax.set_title(f'{REF_BIKE} → {other}')
        ax.legend(fontsize=9)

    plt.suptitle('Grade decomposition — is the effect aero (flat) or weight (steep)?', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No grade column in deltas — skipping grade decomposition.')